<p align="center">
<a href="https://duckietown.com"><img src="../assets/images/dtlogo.png" alt="Duckietown Logo" width="30%"></a>
</p>


<style>
.lx-table {
  margin: 1.5em auto;
  text-align: left;
}

.lx-table caption {
  caption-side: top;
  font-size: 0.9em;
  margin-bottom: 0.6em;
  text-align: center;
}

.lx-figure {
  margin: 1.5em auto;
  text-align: center;
}

.lx-figure figcaption {
  font-size: 0.9em;
  margin-top: 0.6em;
  text-align: center;
}

table {
  margin: 0 auto 1.5em;
}

p:has(> a[id^='table-']) {
  margin: 0;
}

p:has(> a[id^='table-']) + p,
a[id^='table-'] + p {
  font-size: 0.9em;
  margin: 1.5em 0 0.6em;
  text-align: center;
}

p.lx-figure {
  margin: 1.5em auto 0;
}

p.lx-figure + p {
  font-size: 0.9em;
  margin: 0.6em 0 1.5em;
  text-align: center;
}
</style>

# Quaternions and Rotation Representations

The preceding notebook showed why Euler angles are convenient for interpreting attitude but become ambiguous at gimbal lock. Rotation matrices remain well defined there, although storing their nine constrained entries is more than an orientation estimate needs. The four-component `orientation` field encountered in [Notebook 1](./1-imu-basics.ipynb#gathering-imu-measurements-from-a-duckiedrone) suggests another possibility: a unit quaternion represents the same rotation compactly, without an Euler-angle singularity.

This notebook develops that representation from an axis and a rotation angle, then connects its algebra to composing rotations and displaying Euler angles.

This notebook addresses the following questions:

- How does a unit quaternion encode an axis and angle of rotation?

- In which order do quaternion products apply rotations?

- How is a quaternion converted to Euler angles, and what happens at gimbal lock?

## Quaternions: the practical way out

Unit quaternions use four constrained components to represent the three rotational degrees of freedom across the full range of orientations.

A hovering Duckiedrone stays far from $\beta=\pm90^\circ$, so why does gimbal lock matter? If orientation is stored as Euler angles, the body-frame angular-velocity components measured by the gyroscope must first be converted into roll, pitch, and yaw rates. For the ZYX convention, that conversion contains terms divided by $\cos(\beta)$, where $\beta$ is pitch, and is singular at $\beta=\pm90^\circ$.

The trouble starts earlier: at $\beta=\pm80^\circ$, gyroscope noise is already amplified almost six times on its way into yaw, since $1/\cos(80^\circ)\approx5.8$. The estimator also has to work at every orientation, for example after a crash or while the Duckiedrone is turned over by hand.

Changing the Euler-angle order moves the singular orientations, but does not remove them. A quaternion stores four components, compared with the nine entries of a rotation matrix, making it a more compact representation for estimation and communication of angles and rotations.

### What a quaternion is

Notebook 3 constructed an orientation from three elementary rotations. [Euler's rotation theorem](https://en.wikipedia.org/wiki/Euler%27s_rotation_theorem) gives another description: any proper three-dimensional rotation can be represented by a single rotation about an axis. A rotation quaternion encodes that axis and angle through four related components.

[Table 1](#table-1) collects the notation used in this notebook.

<a id="table-1"></a>

Table 1: Summary of symbols used for quaternions in this notebook.

| Symbol | Meaning | Units |
| --- | --- | --- |
| $\mathbf{u}=(u_x,u_y,u_z)^{\mathsf{T}}$ | Unit vector along the rotation axis, $\lVert\mathbf{u}\rVert=1$ | Dimensionless |
| $\psi$ | Rotation angle about $\mathbf{u}$ | $\mathrm{rad}$ |
| $q=(q_x,q_y,q_z,q_w)$ | Unit quaternion, in the scalar-last order of the ROS 2 message | Dimensionless |
| $\mathbf{q}_{\mathrm{v}}=(q_x,q_y,q_z)^{\mathsf{T}}$ | Vector part of $q$ | Dimensionless |
| $q_w$ | Scalar part of $q$ | Dimensionless |
| $\alpha$, $\beta$, $\gamma$ | Roll, pitch, and yaw in the ZYX convention of Notebook 3 | $\mathrm{rad}$ |

Let $\mathbf{u}$ be the unit vector along that axis and $\psi$ the rotation angle. A quaternion combines a three-component vector part, $\mathbf{q}_{\mathrm{v}}$, with a scalar part, $q_w$. This notebook uses the scalar-last order of the ROS 2 message:

$$
q = (\mathbf{q}_{\mathrm{v}},q_w) = (q_x,q_y,q_z,q_w).
$$

A rotation through $\psi$ about $\mathbf{u}$ is represented by choosing

$$
\mathbf{q}_{\mathrm{v}}=\mathbf{u}\sin\left(\frac{\psi}{2}\right),
\qquad
q_w=\cos\left(\frac{\psi}{2}\right).
$$

Together, these components give the axis-angle representation

$$
q =
\left(
\mathbf{u}\sin\left(\frac{\psi}{2}\right),
\cos\left(\frac{\psi}{2}\right)
\right).
$$

Here, $q$ represents the same body-to-world rotation as $\mathbf{R}_{\mathrm{WB}}$ in Notebook 3.

[Figure 1](#figure-1) connects the quaternion components to the axis and angle.

<a id="figure-1"></a>

<p class="lx-figure" style="width:420px;">
  <img src="../assets/_images/quaternion_axis_angle.png" alt="A rotation axis u and angle psi, with the quaternion vector and scalar parts labeled." style="width:100%;" />
</p>

Figure 1: The vector part stores $\mathbf{u}\sin(\psi/2)$, and the scalar part stores $\cos(\psi/2)$. The labels $x$, $y$, $z$, and $w$ beneath the drawing denote quaternion components, rather than point coordinates.

With no rotation, $\psi=0$, the vector part vanishes and the scalar part is one:

$$
q=(0,0,0,1).
$$

This is the identity quaternion. The axis can be chosen arbitrarily because a zero-angle turn about any axis leaves the orientation unchanged.

For a positive quarter turn about $z$, the axis is $\mathbf{u}=(0,0,1)^{\mathsf{T}}$ and $\psi=90^\circ$. Therefore:

$$
q=
\left(0,0,\sin45^\circ,\cos45^\circ\right)
=
\left(0,0,\frac{\sqrt{2}}{2},\frac{\sqrt{2}}{2}\right)
\approx(0,0,0.7071,0.7071).
$$

This is the yaw rotation that points the nose along world $+y$ in Notebook 3. The component values are sines and cosines of a half-angle, so reading them directly as roll, pitch, or yaw would give the wrong interpretation.

### A few properties of quaternions

1. A unit quaternion satisfies:

$$
\lVert q\rVert^2
=
q_x^2+q_y^2+q_z^2+q_w^2
=
1.
$$

Any axis-angle quaternion satisfies it, because $\lVert\mathbf{u}\rVert=1$. This constraint means that its four components do not describe four independent rotational degrees of freedom. Numerical calculations can introduce small deviations from unit length, which normalization corrects by dividing all four components by $\lVert q\rVert$. The all-zero quaternion has zero length and cannot be normalized into an orientation; it must not be confused with the identity quaternion.

2. A small turn always changes the four numbers only a little, whatever the orientation, so nothing goes wrong at $\beta=\pm90^\circ$, where Euler angles break.


3. Each orientation nevertheless has two quaternion representations: $q$ and $-q$. Their components have opposite signs, but they describe exactly the same rotation.


4. Successive rotations are represented by quaternion multiplication: $q_2q_1$ applies $q_1$ first, then $q_2$. As with rotation matrices, the order matters. The [quaternion notes](../assets/data/notes-on-quaternions-algebra.md) derive this from the quaternion product.

### Converting to Euler angles

Roll, pitch, and yaw make a quaternion orientation easier to interpret on a display, provided the conversion uses the same frame and angle conventions.

Assume a unit quaternion $q=(q_x,q_y,q_z,q_w)$ representing the body-to-world rotation in the ZYX convention of [Equation (4) in Notebook 3](./3-rotation-representations.ipynb#equation-4), and roll ($\alpha$), pitch ($\beta$), and yaw ($\gamma$) being expressed in radians.

Here, $\arcsin$ denotes inverse sine, and $\operatorname{atan2}(y,x)$ uses the signs of both arguments to determine the quadrant. Away from gimbal lock, the roll angle is:

<span id="equation-1"></span>

$$
\alpha
=
\operatorname{atan2}\big(
2(q_wq_x+q_yq_z),\;
1-2(q_x^2+q_y^2)
\big).
\tag{1}
$$

The pitch angle is:

<span id="equation-2"></span>

$$
\beta
=
\arcsin\big(2(q_wq_y-q_zq_x)\big).
\tag{2}
$$

The yaw angle is:

<span id="equation-3"></span>

$$
\gamma
=
\operatorname{atan2}\big(
2(q_wq_z+q_xq_y),\;
1-2(q_y^2+q_z^2)
\big).
\tag{3}
$$

Substituting the exact quarter-turn quaternion above gives $\alpha=0$, $\beta=0$, and $\gamma=\pi/2$. Negating all four quaternion components gives the same results because each expression uses products or squares of the components.

[Equation (2)](#equation-2) selects pitch in the range $[-\pi/2,\pi/2]$. At either endpoint, both arguments of $\operatorname{atan2}$ in [Equation (1)](#equation-1) and [Equation (3)](#equation-3) become zero in exact arithmetic. Roll and yaw are then inseparable, as demonstrated in Notebook 3. The quaternion remains valid, while conversion to Euler angles requires choosing one of the equivalent angle triples.

The helper `quaternion_to_euler` in [imu_math.py](../packages/solution/solution/imu_math.py), used by [Notebook 7](./7-interfacing-with-imu.ipynb), implements these formulas; at the pole, it sets roll to 0 and puts the whole turn into yaw.

ROS 2 stores the orientation in `sensor_msgs/Imu` using the following quaternion fields:

```text
geometry_msgs/Quaternion orientation
    float64 x
    float64 y
    float64 z
    float64 w
```

Some interfaces use scalar-first order instead, so the order must be checked at each interface. Reordering components alone does not convert between different reference frames.

## Comparing rotation representations

[Table 2](#table-2) compares the representations developed in Notebooks 3 and 4.

<a id="table-2"></a>

Table 2: Comparison of three common rotation representations.

| Representation | Stored values | Constraint or ambiguity | Typical use |
| --- | ---: | --- | --- |
| ZYX Euler angles | 3 | Roll and yaw become dependent at $\beta=\pm90^\circ$; angle values can also wrap at range boundaries | Interpreting orientation on a display |
| Rotation matrix | 9 | $\mathbf{R}^{\mathsf{T}}\mathbf{R}=\mathbf{I}$ and $\det(\mathbf{R})=+1$; no Euler-angle singularity | Transforming vector components between frames |
| Unit quaternion | 4 | $\lVert q\rVert=1$ and $q$ represents the same orientation as $-q$; no Euler-angle singularity | Storing, updating, and transmitting orientation |

All three describe the same three rotational degrees of freedom. Matrices and unit quaternions use constrained components to represent orientation across the full range of rotations, while Euler angles provide a familiar interpretation within their coordinate limitations.

## Summary and next steps

A unit quaternion encodes an orientation through four constrained components. Its axis-angle form explains the identity and quarter-turn examples, while its sign ambiguity explains why opposite component values can describe the same attitude.

Conversion to roll, pitch, and yaw requires a stated frame convention, component order, and valid unit quaternion. The Euler-angle singularity returns during that conversion even though the quaternion remains well defined.

[Notebook 5](./5-duckiedrone-imu-architecture.ipynb) follows the measurements and orientation estimate through the Duckiedrone's flight controller and into ROS 2, identifying where their reference-frame conventions change.

## Further reading

[Quaternion fundamentals](https://docs.ros.org/en/jazzy/Tutorials/Intermediate/Tf2/Quaternion-Fundamentals.html) explains quaternion component order, normalization, and rotation composition in ROS 2.

The video [Quaternions and 3d rotation, explained interactively](https://www.youtube.com/watch?v=zjMuIxRvygQ) develops the geometry behind the half-angle. Successive rotations can also be composed by quaternion multiplication; [Visualizing quaternions](https://eater.net/quaternions) explores that algebra interactively.

## Checkpoint

Run the self-check in the next cell. Write or select a response before revealing the answer.


In [ ]:
import sys
from pathlib import Path

working_directory = Path.cwd()
parent_directory = working_directory.parent
if (parent_directory / "packages").is_dir():
    parent_directory_path = str(parent_directory)
    sys.path.insert(0, parent_directory_path)

from packages.checkpoint_self_check import display_checkpoint_self_checks

display_checkpoint_self_checks()